In [ ]:
import torch

import torch.nn as nn

# Create a simple input tensor (batch=1, channel=1, height=7, width=7)
input_tensor = torch.arange(49, dtype=torch.float32).view(1, 1, 7, 7)
print("Input tensor:")
print(input_tensor.shape)
print(input_tensor)

# Define a Conv2d layer without dilation (dilation=1)
conv_no_dilation = nn.Conv2d(in_channels=1, out_channels=1, kernel_size=3, dilation=1, padding=1, bias=False)

# Define a Conv2d layer with dilation=2.
# For a 3x3 kernel with dilation=2, the effective kernel size becomes 5x5 (2*(3-1)+1),
# so we set padding=2 to keep the output size the same.
conv_dilated = nn.Conv2d(in_channels=1, out_channels=1, kernel_size=3, dilation=2, padding=2, bias=False)

# Initialize both convolutional layers with constant weights for clarity.
with torch.no_grad():
  custom_kernel = torch.tensor([[[[1.0, 2.0, 3.0],
                                  [4.0, 5.0, 6.0],
                                  [7.0, 8.0, 9.0]]]]) # shape: (1, 1, 3, 3)
  conv_no_dilation.weight.copy_(custom_kernel)
  conv_dilated.weight.copy_(custom_kernel)

  print("\nConv2d with dilation=1 Weights:")
  print(conv_no_dilation.weight.shape)
  print(conv_no_dilation.weight)

  print("\nConv2d with dilation=2 Weights:")
  print(conv_dilated.weight.shape)
  print(conv_dilated.weight)

# Compute outputs
output_no_dilation = conv_no_dilation(input_tensor)
output_dilated = conv_dilated(input_tensor)

print("\nOutput with dilation=1:")
print(output_no_dilation.shape)
print(output_no_dilation)
# output[0, 0, 0, 0] = input_tensor[0, 0, 0:3, 0:3] * custom_kernel = 134
# output[0, 0, 0, 0] = 0 * 1 + 0 * 2 + 0 * 3 + 0 * 4 + 0 * 5 + 1 * 6 + 0 * 7 + 7 * 8 + 8 * 9 = 134

print("\nOutput with dilation=2:")
print(output_dilated.shape)
print(output_dilated)
# output[0, 0, 0 0] = 0 * 1 + 0 + 2 + 0 + 14 + 16 = 32

Input tensor:
torch.Size([1, 1, 7, 7])
tensor([[[[ 0.,  1.,  2.,  3.,  4.,  5.,  6.],
          [ 7.,  8.,  9., 10., 11., 12., 13.],
          [14., 15., 16., 17., 18., 19., 20.],
          [21., 22., 23., 24., 25., 26., 27.],
          [28., 29., 30., 31., 32., 33., 34.],
          [35., 36., 37., 38., 39., 40., 41.],
          [42., 43., 44., 45., 46., 47., 48.]]]])

Conv2d with dilation=1 Weights:
torch.Size([1, 1, 3, 3])
Parameter containing:
tensor([[[[1., 2., 3.],
          [4., 5., 6.],
          [7., 8., 9.]]]], requires_grad=True)

Conv2d with dilation=2 Weights:
torch.Size([1, 1, 3, 3])
Parameter containing:
tensor([[[[1., 2., 3.],
          [4., 5., 6.],
          [7., 8., 9.]]]], requires_grad=True)

Output with dilation=1:
torch.Size([1, 1, 7, 7])
tensor([[[[ 134.,  211.,  250.,  289.,  328.,  367.,  238.],
          [ 333.,  492.,  537.,  582.,  627.,  672.,  423.],
          [ 564.,  807.,  852.,  897.,  942.,  987.,  612.],
          [ 795., 1122., 1167., 1212., 1257., 

In [59]:
import torch
import torch.nn.functional as F

input_tensor = torch.arange(49, dtype=torch.float32).view(1, 1, 7, 7)
print(f"Input tensor:\n{input_tensor}")

custom_kernel = torch.tensor([[[[1.0, 2.0, 3.0],
                                [4.0, 5.0, 6.0],
                                [7.0, 8.0, 9.0]]]]) # shape: (1, 1, 3, 3)
print(f"Custom kernel:\n{custom_kernel}")

padded_input = F.pad(input_tensor, (1, 1, 1, 1), value=0)
receptive_field = padded_input[0, 0, 0:3, 0:3]
output0000 = torch.sum(receptive_field * custom_kernel)
# element-wise multiplication and sum
print(f"receptive_field:\n{receptive_field}")
print(f"output[0,0,0,0]: {output0000}")

padded_input = F.pad(input_tensor, (2, 2, 2, 2), value=0)
receptive_field = padded_input[0, 0, 0:5:2, 0:5:2]
output_dilated0000 = torch.sum(receptive_field * custom_kernel)
# element-wise multiplication and sum
print(f"receptive_field:\n{receptive_field}")
print(f"output_dilated[0,0,0,0]: {output_dilated0000}")

Input tensor:
tensor([[[[ 0.,  1.,  2.,  3.,  4.,  5.,  6.],
          [ 7.,  8.,  9., 10., 11., 12., 13.],
          [14., 15., 16., 17., 18., 19., 20.],
          [21., 22., 23., 24., 25., 26., 27.],
          [28., 29., 30., 31., 32., 33., 34.],
          [35., 36., 37., 38., 39., 40., 41.],
          [42., 43., 44., 45., 46., 47., 48.]]]])
Custom kernel:
tensor([[[[1., 2., 3.],
          [4., 5., 6.],
          [7., 8., 9.]]]])
receptive_field:
tensor([[0., 0., 0.],
        [0., 0., 1.],
        [0., 7., 8.]])
output[0,0,0,0]: 134.0
receptive_field:
tensor([[ 0.,  0.,  0.],
        [ 0.,  0.,  2.],
        [ 0., 14., 16.]])
output_dilated[0,0,0,0]: 268.0
